# Modern Application Development – I: Comprehensive Lecture Notes  
**Professor Nitin Chandrachoodan, IIT Madras**  
**Week 12: Deployment, Cloud Services, CI/CD, Containers & Course Conclusion**

---

## Table of Contents

1. [Introduction to Deployment](#1-introduction-to-deployment)
2. [The Architecture of a Deployed Application](#2-architecture-of-a-deployed-application)
3. [The Service Approach: IaaS, PaaS, SaaS](#3-the-service-approach)
4. [Version Control and Git](#4-version-control-and-git)
5. [Continuous Integration and Continuous Delivery/Deployment (CI/CD)](#5-cicd)
6. [Containers and Orchestration](#6-containers-and-orchestration)
7. [Course Summary: Tying Everything Together](#7-course-summary)

---

## 1. Introduction to Deployment

After weeks of designing, coding, and testing a web application on your local machine, the final step is to **deploy** it—make it available on the internet so that real users can access it. Deployment is much more than copying files to a server; it involves decisions about infrastructure, scaling, security, automation, and maintenance.

The lecture outlines the journey of an application from an idea to a publicly accessible service:

1. **Idea and local development:** You conceive an app, write code on your laptop or desktop using editors and tools. The app runs locally, perhaps using Flask's built‑in development server. All files (source code, templates, static assets) reside on your machine. This is fine for development, but it has severe limitations:
   - Your machine is not always on.
   - It lacks a permanent public IP address.
   - It cannot handle many concurrent users.
   - Power or network interruptions make the app unavailable.
   - Running databases, web servers, and other services on a single machine can cause resource contention.

2. **The need for permanent deployment:** To serve users reliably, you need a **dedicated server** or a set of servers that are:
   - Always on.
   - Connected to a high‑bandwidth internet link with a static IP or domain name.
   - Supplied with uninterrupted power and cooling.
   - Monitored and maintained.

   Such facilities are called **data centers**. A data center is a specialised building housing thousands of computers, with redundant power (generators, UPS), sophisticated cooling, physical security, and multiple high‑speed network connections. For an individual developer or a small company, building and maintaining a private data center is prohibitively expensive and complex.

3. **The rise of the Cloud:** The **cloud** abstracted away the physical infrastructure. Cloud providers (Amazon Web Services, Google Cloud, Microsoft Azure, DigitalOcean, Linode) own and operate massive data centers. They rent out virtualised computing resources on a pay‑as‑you‑go basis. This democratised access to enterprise‑grade infrastructure: anyone with a credit card can launch a virtual server in minutes.

The cloud is not just about renting machines; it is a whole ecosystem of services that handle storage, networking, load balancing, monitoring, and more. This shift fundamentally changed how applications are deployed and scaled.

---

## 2. The Architecture of a Deployed Application

Deploying an application is rarely as simple as putting everything on one server. As user demand grows, the application must **scale** – handle more requests per second without degrading performance. The lecture walks through a typical evolution of a web application’s infrastructure.

### 2.1 The Simplest Case: One Server
Initially, a single server runs everything: the web server (e.g., Nginx, Apache), the application logic (Flask/Python), and the database (MySQL, PostgreSQL). This is fine for low traffic or internal tools, but it has no redundancy and limited capacity.

### 2.2 Splitting Frontend and Backend
As load increases, the first step is often to separate the **web/application server** from the **database server**. This allows each to be tuned independently:
- The web server needs more CPU and network I/O.
- The database server needs fast disks (SSD) and large RAM for caching.

The two servers communicate over a private, high‑speed internal network, not the public internet. This separation also improves security: the database is not directly exposed to the outside world.

### 2.3 Adding a Load Balancer
Eventually, a single application server becomes a bottleneck. To handle more traffic, you can run **multiple identical application servers** behind a **load balancer**. The load balancer is a reverse proxy that receives all incoming requests from users and distributes them among the available backend servers according to a scheduling algorithm (round‑robin, least connections, etc.).

**Key benefits:**
- **Scalability:** Add more backend servers as needed (horizontal scaling).
- **High availability:** If one backend server fails, the load balancer routes traffic to the healthy ones.
- **Transparency:** Users only ever see the load balancer’s IP address; the internal topology is hidden.
- **SSL termination:** The load balancer can handle HTTPS encryption/decryption, relieving backend servers of that CPU overhead.

### 2.4 Additional Middleware and Specialised Services
The architecture can be further refined by adding dedicated services, each as a separate server or set of servers:

- **HTTPS / SSL terminator:** A front‑end proxy that handles TLS certificates and encryption.
- **Logging server:** Collects and aggregates logs from all components for monitoring and debugging.
- **Caching server:** e.g., Redis or Memcached, to store frequently accessed data in memory, reducing database load.
- **Content Delivery Network (CDN):** A geographically distributed network of servers that caches static assets (images, CSS, JavaScript, fonts) closer to end‑users. When a user requests a static file, the CDN edge node serves it directly, reducing latency and bandwidth on the origin server. Popular CDNs include Cloudflare, Akamai, and Amazon CloudFront.

### 2.5 Database Scaling
Databases are harder to scale horizontally. Common strategies include:
- **Read replicas:** One primary database handles writes; multiple read‑only replicas serve read queries, synchronised asynchronously.
- **Sharding:** Partitioning data across multiple database servers based on a key (e.g., user ID).
- **NoSQL databases:** Designed from the ground up for horizontal scaling and eventual consistency (covered in earlier weeks).

The lecture’s architectural diagram summarises how a modern, scalable web application might look: user → CDN → load balancer → multiple web servers → multiple database servers, with logging and caching services on the side. The user sees only a single domain name, but behind it lies a complex, elastic infrastructure managed by cloud services.

---

## 3. The Service Approach: IaaS, PaaS, SaaS

The cloud is often described in terms of **service models**, which abstract away different layers of the technology stack. The lecture introduces the three classic models.

### 3.1 Infrastructure as a Service (IaaS)
IaaS provides virtualised computing resources over the internet. The provider manages the physical hardware (servers, storage, networking), while you are responsible for everything from the operating system upwards: installing the OS, applying security patches, configuring software, and managing your applications.

- **Examples:** Amazon EC2, Google Compute Engine, Microsoft Azure Virtual Machines, DigitalOcean Droplets.
- **You control:** OS, middleware, runtime, application, data.
- **Provider controls:** Virtualisation, servers, storage, networking.
- **Analogy:** Renting a plot of land and building your own house; you are responsible for construction, plumbing, electricity, but the land is provided.

### 3.2 Platform as a Service (PaaS)
PaaS abstracts away the operating system and middleware. The provider gives you a managed environment where you can deploy your application code. You do not worry about server maintenance, OS updates, or scaling the underlying infrastructure; you just push your code.

- **Examples:** Google App Engine, Heroku, AWS Elastic Beanstalk, Replit (for development), Glitch.
- **You control:** Application code and configuration.
- **Provider controls:** OS, web server, runtime (e.g., Python, Node.js), scaling, load balancing, monitoring.
- **Analogy:** Renting a fully furnished apartment; you bring your belongings (code) and live in it, but the building management handles repairs, security, and utilities.

**PaaS in detail:** The lecture demonstrates several PaaS examples:
- **Replit:** Primarily an educational and collaborative development environment. It provides a browser‑based code editor, a shell, and automatically hosts your Flask (or other) app at a URL. Behind the scenes, it provisions a Linux environment with Python, Flask, and dependencies. However, Replit is not designed for production deployment; apps sleep after inactivity and have resource limits.
- **Glitch:** Similar to Replit but more focused on community and sharing apps. It allows you to “remix” projects and deploy small web apps quickly. Like Replit, it abstracts away the server management.
- **Google App Engine (GAE):** A production‑grade PaaS from Google Cloud. You write your application (e.g., Flask), define a configuration file (`app.yaml`), and deploy with a single command (`gcloud app deploy`). GAE automatically scales your app based on traffic, manages load balancing, and provides logging and monitoring. It is a stark contrast to Replit/Glitch: the interface is more complex, reflecting its suitability for serious, scalable applications. GAE also offers a free tier for low‑traffic apps, making it accessible for experimentation.

### 3.3 Software as a Service (SaaS)
SaaS delivers fully functional software applications over the internet, typically on a subscription basis. The user does not manage any infrastructure or code; they simply use the software through a web browser.

- **Examples:** Gmail, Google Docs, Microsoft Office 365, Salesforce, GitHub, GitLab.com.
- **You control:** Your data and user‑specific settings.
- **Provider controls:** Everything else – application, runtime, security, infrastructure.
- **Analogy:** Eating at a restaurant; you consume the meal, but the kitchen, ingredients, and chef are entirely managed by the restaurant.

### 3.4 Choosing the Right Service Model
The choice depends on the level of control vs. convenience needed:
- **IaaS** gives maximum flexibility and control, suitable for custom environments or applications with specific OS requirements.
- **PaaS** speeds up development and deployment, ideal for standard web applications where you want to focus on code, not servers.
- **SaaS** is for end‑users who just need the software to work without any development effort.

In modern cloud development, many applications use a mix: a PaaS for the web frontend, a managed database service (which is itself a form of PaaS), and IaaS for custom background processing.

---

## 4. Version Control and Git

Deployment is not a one‑time event; it is a continuous process. As the application evolves, you need a robust way to manage changes to the source code. This is where **version control systems (VCS)** come in.

### 4.1 Why Version Control?
Writing software is iterative. You add features, fix bugs, refactor code. Without version control:
- You have to manually keep copies of old code (e.g., `myapp_v1.py`, `myapp_v2_backup.py`).
- It is hard to know what changed, when, and why.
- Collaborating with multiple developers becomes a nightmare.
- Rolling back to a previous working state after a bad change is painful.

A VCS tracks every modification to the codebase over time. It records who made the change, when, and (via commit messages) why. It allows you to rewind to any previous state, compare versions, and merge work from different people.

### 4.2 Git: The Distributed Version Control System
**Git** is by far the most popular VCS today. It is **distributed**: every developer has a complete copy of the repository (all history, all branches) on their local machine. This contrasts with older centralised systems like SVN, where a single server holds the canonical version.

**Key concepts in Git:**
- **Repository (repo):** A directory containing your project’s files and the entire history of changes (stored in a hidden `.git` folder).
- **Commit:** A snapshot of the repository at a point in time. Each commit has a unique hash (SHA‑1) and a parent commit(s), forming a directed acyclic graph.
- **Branch:** A lightweight, movable pointer to a commit. Branches allow you to work on features or fixes in isolation without affecting the main codebase. The default branch is typically called `main` or `master`.
- **Merge:** Combining the changes from one branch into another. Git can automatically merge changes if they do not conflict; if two developers edited the same lines, a merge conflict must be resolved manually.
- **Remote:** A version of the repository hosted on another machine (e.g., GitHub, GitLab). You `push` your local commits to the remote and `pull` others’ commits from it.

### 4.3 A Common Branching Model: Git Flow
The lecture presents a simplified version of the **Git Flow** branching model to illustrate how branches facilitate organised development:

- **`main` (or `master`) branch:** Holds the production‑ready code. Only thoroughly tested, stable releases are merged here.
- **`develop` branch:** The main integration branch for ongoing development. New features are merged into `develop`, not directly into `main`.
- **Feature branches:** Created from `develop` for each new feature or task. Once the feature is complete and tested, it is merged back into `develop`. This keeps `develop` always in a relatively stable state.
- **Release branches:** When `develop` is ready for a release, a release branch is created. This branch is used for final bug fixes, documentation updates, and version bumping. Once stable, it is merged into `main` (and tagged with a version number) and also merged back into `develop` to incorporate any release‑specific fixes.
- **Hotfix branches:** Created from `main` to urgently fix a critical bug in production. The fix is developed and then merged into both `main` and `develop` so the bug is squashed everywhere.

This disciplined approach prevents chaos, ensures that `main` is always deployable, and allows multiple developers to work in parallel without stepping on each other’s toes. Platforms like **GitHub** and **GitLab** provide web interfaces, pull request (or merge request) workflows, code review tools, and issue tracking built on top of Git, making collaboration even smoother.

---

## 5. Continuous Integration and Continuous Delivery/Deployment (CI/CD)

Version control alone is not enough to guarantee quality. Manually integrating code changes, running tests, and deploying is slow and error‑prone. **CI/CD** automates these processes, forming a critical part of modern DevOps practices.

### 5.1 Continuous Integration (CI)
**Continuous Integration** is the practice of frequently (often several times a day) merging all developers’ working copies into a shared mainline branch. Every merge triggers an automated **build** and a comprehensive **test suite**.

**Core principles of CI:**
1. **Automated build:** The code is compiled (if necessary) and packaged. In Python/Flask apps, this might involve checking that dependencies resolve, creating a virtual environment, etc.
2. **Automated testing:** The full test suite (unit tests, integration tests) is executed. If any test fails, the team is immediately notified. The commit that broke the build is identified, and fixing it becomes the top priority.
3. **Fast feedback:** The CI pipeline must run quickly (ideally minutes) so developers are not blocked.
4. **Single source of truth:** The shared repository is always in a known, tested state.

**Benefits:** Bugs are caught early when they are cheaper to fix; integration hell is avoided; developers have confidence that their changes haven’t broken existing functionality; the codebase is always ready for release.

**Tools:** Jenkins, GitHub Actions, GitLab CI/CD, CircleCI, Travis CI.

### 5.2 Continuous Delivery (CD) and Continuous Deployment
**Continuous Delivery** extends CI: after the code passes all tests, it is automatically prepared for a release, but the actual deployment to production is a manual decision (e.g., a button click). The software can be released at any time with confidence.

**Continuous Deployment** goes one step further: every change that passes all stages of the pipeline is automatically deployed to production without human intervention. This is only suitable for highly mature teams with extremely robust testing and monitoring, because a bug that slips past the tests will instantly reach users.

The lecture highlights the difference with a concrete example:
- **Nightly builds** (a form of continuous delivery): a desktop application like LibreOffice might automatically build a new version every night from the latest `develop` branch. Beta testers can download and test it, but the stable release channel remains separate.
- **Web app continuous deployment:** A Flask app on Google App Engine might be re‑deployed on every push to `main`. The moment the tests pass, the new version goes live. This enables rapid iteration and instant bug fixes, but demands rigorous test coverage.

**CI/CD pipeline stages** typically include: linting (code style checks), unit tests, integration tests, security scans, building the application package, deploying to a staging environment for further testing, and finally deploying to production.

---

## 6. Containers and Orchestration

### 6.1 The Problem with Traditional Deployment
Deploying an application on a server often leads to dependency conflicts. If two applications require different versions of Python or conflicting system libraries, managing them on the same OS is a headache. Virtual machines (VMs) solve this by emulating entire hardware, each running its own OS, but VMs are heavy: each one consumes significant memory and disk space for the full OS, and they boot slowly.

### 6.2 What Are Containers?
**Containers** provide a lightweight alternative. A container is an isolated environment that shares the host operating system’s kernel but runs as if it has its own dedicated OS. They achieve isolation through Linux kernel features:

- **Namespaces:** Provide isolated views of system resources. A process in a container sees its own file system, process tree, network interfaces, and user IDs, completely separate from the host and other containers.
- **Control Groups (cgroups):** Limit and account for resource usage (CPU, memory, disk I/O) per container, preventing a single container from monopolising the host’s resources.

**Benefits of containers:**
- **Lightweight:** They do not contain a full OS; they share the host kernel. They start in seconds and have minimal overhead.
- **Portable:** A container image packages the application and all its dependencies (libraries, binaries, configuration). The same image runs identically on a developer’s laptop, a test server, or in production.
- **Reproducible:** Since the environment is frozen in the image, “it works on my machine” problems are virtually eliminated.
- **Sandboxed:** A process inside a container cannot easily affect other containers or the host.

### 6.3 Docker
**Docker** is the most popular platform for building, shipping, and running containers. A **Dockerfile** is a text file containing instructions to build a container image:

```dockerfile
FROM python:3.9-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install -r requirements.txt
COPY . .
CMD ["python", "app.py"]
```
- `FROM` specifies a base image (official Python image).
- `COPY` adds files from the host into the image.
- `RUN` executes commands during the build (e.g., install dependencies).
- `CMD` defines the default command to run when the container starts.

The result is an **image** – a read‑only template. Running an image creates a **container** instance. Docker also provides tools for networking containers together, managing volumes for persistent data, and a public registry (Docker Hub) to share images.

### 6.4 Orchestration: Managing Containers at Scale
A real application comprises multiple services (web server, database, cache, worker). Instead of managing each container manually, **orchestration** tools automate the deployment, scaling, and management of containerised applications.

- **Docker Compose:** A simpler tool for defining and running multi‑container Docker applications on a single host. You write a `docker-compose.yml` file specifying services, networks, and volumes. With a single command (`docker-compose up`), the entire application stack is started. Ideal for development and small deployments.

- **Kubernetes (K8s):** The industry standard for container orchestration at scale. It manages clusters of machines (nodes) and schedules containers (pods) across them. Kubernetes provides:
  - **Service discovery and load balancing:** Automatically assigns DNS names and distributes traffic.
  - **Self‑healing:** Restarts failed containers, replaces and reschedules them on healthy nodes.
  - **Horizontal scaling:** Automatically scales the number of pods based on CPU or custom metrics.
  - **Rolling updates and rollbacks:** Updates the application with zero downtime; can roll back if errors occur.
  - **Secret and configuration management:** Stores sensitive data and configs separately from the image.

Kubernetes is complex to set up and manage, but cloud providers offer managed Kubernetes services (Google GKE, Amazon EKS, Azure AKS) that simplify operations. For large‑scale, dynamic applications, it is an invaluable tool.

The lecture situates containers and orchestration as a key enabler for the scalable, resilient architectures discussed earlier. Each box in the architectural diagram (load balancer, web server, database) could be a container, and Kubernetes can manage their entire lifecycle.

---

## 7. Course Summary: Tying Everything Together

The final lecture provides a concise, high‑level summary of the entire course, mapping all the concepts into a complete picture of modern application development.

### 7.1 The Overall Development and Deployment Lifecycle
1. **Idea and Requirements:** Understand what problem the app solves. Define user stories and functional/non‑functional requirements.
2. **Design and Architecture:** Model the data (entities, relationships), design the API (RESTful, OpenAPI), and plan the user interface (wireframes, HTML/CSS/JS).
3. **Implementation:** Write the code following the MVC pattern.
   - **Frontend:** HTML, CSS, JavaScript (possibly with frameworks like React/Vue, but in this course we focused on server‑rendered templates with Jinja2 and Bootstrap).
   - **Backend:** Python/Flask handling routes (controllers), using SQLAlchemy as ORM (model).
   - **Database:** SQLite for development, with a path to production databases like PostgreSQL.
4. **Testing:** Write unit tests, integration tests. Use pytest. Achieve high coverage; automate testing in CI pipeline.
5. **Version Control:** Track all code changes with Git. Collaborate via pull requests.
6. **Deployment:**
   - **Local development:** everything on one machine.
   - **Production:** deploy to a cloud platform (PaaS like Google App Engine or IaaS like a VM), using CI/CD to automate deployment.
   - **Scale:** add load balancers, multiple instances, CDN, caching, database replicas as traffic grows.
   - **Containerise:** package the app and its dependencies in Docker containers; use Kubernetes for orchestration if at massive scale.

### 7.2 Core Technologies Covered
- **HTTP and the Web:** The foundation. Request‑response model, HTTP methods, status codes, REST principles, statelessness.
- **Markup and Styling:** HTML5 for structure, CSS3 (plus Bootstrap) for responsive design.
- **Client‑side Scripting:** JavaScript for DOM manipulation, asynchronous updates (AJAX/fetch), custom elements, and modern frameworks.
- **Backend Programming:** Python with Flask. Routing, controllers, Jinja2 templating, session management, authentication.
- **Databases:** Relational model, SQL, SQLAlchemy ORM, migrations, indexing, transactions. Introduction to NoSQL alternatives.
- **APIs:** Designing and documenting RESTful APIs with OpenAPI/Swagger. Building APIs with Flask‑RESTful.
- **Security:** Access control (RBAC, least privilege), authentication (passwords, tokens, OAuth), session management, HTTPS/TLS, SQL injection prevention, logging.
- **Testing:** PyTest, unit/integration testing, code coverage, TDD, regression testing.
- **Deployment and DevOps:** Cloud service models, Git, CI/CD, containers (Docker), orchestration (Kubernetes).

### 7.3 The Layered View of a Web Application
The course can be visualised as layers:
1. **Presentation Layer (Frontend):** The user’s browser. HTML, CSS, JavaScript. This is what the user interacts with.
2. **Application Logic Layer (Backend/Controller):** Flask routes, business logic. Runs on the server. Processes requests, makes decisions, orchestrates data flow.
3. **Data Layer (Model):** Database (SQLite, PostgreSQL, etc.). Persistent storage of all application data.
4. **Infrastructure Layer:** The servers, network, cloud services that run the application. Includes load balancers, CDNs, containers.

Cross‑cutting concerns: **Security** (present at every layer), **Testing** (validates each layer and their integration), and **DevOps** (automates the path from code to production).

### 7.4 Key Takeaways
- **Separation of Concerns:** MVC, HTML vs. CSS vs. JS, API design – all are about isolating different responsibilities so that the system is modular, testable, and maintainable.
- **Statelessness and REST:** The web is inherently stateless; we add sessions and tokens on top. RESTful design embraces this for scalability.
- **Design for Scale:** From the beginning, think about how data is modelled, indexed, and queried. Understand the trade‑offs of SQL vs. NoSQL. Design APIs that can be cached and load‑balanced.
- **Security is not optional:** Validate all inputs on the server, hash passwords, use HTTPS, protect against SQL injection and CSRF, follow the principle of least privilege.
- **Automation:** Manual processes don’t scale. Automate testing (CI), automate deployment (CD), use infrastructure as code, and containerise for consistency.
- **The fundamentals endure:** Frameworks and tools change rapidly, but the underlying principles of HTTP, databases, security, and distributed systems remain constant. Master the fundamentals, and you can adapt to any new technology.

---

This concludes the lecture series. You now have a comprehensive understanding of the entire lifecycle of a modern web application, from concept to deployment, along with the deep theoretical and practical knowledge required to build, secure, and scale robust applications.